# Test FPT_Policy_GraphRAG

Notebook này được dùng để test các truy vấn nhanh vào hệ thống GraphRAG. Bạn có thể chạy từng cell để quan sát kết quả của Vector Search, Graph Search hoặc Hybrid.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from langchain_core.messages import HumanMessage
from src.core.graph import super_agent

In [ ]:
def ask_agent(question: str):
    print(f"\n{'='*60}\nCÂU HỎI: {question}\n{'='*60}")
    initial_state = {
        "messages": [HumanMessage(content=question)],
        "current_question": question,
        "retrieved_info": "",
        "final_answer": ""
    }
    try:
        final_state = super_agent.invoke(initial_state)
        print("\n=> [TRẢ LỜI]:\n")
        print(final_state["final_answer"])
    except Exception as e:
        print(f"\n[LỖI]: {e}")

In [ ]:
# Test 1: Vector Search (Hỏi về khái niệm chung hoặc quy định cụ thể)
ask_agent("Quy định về bảo vệ dữ liệu cá nhân của FPT Software là gì?")

In [ ]:
# Test 2: Graph Search (Hỏi về cấu trúc phân quyền, cấu trúc chức vụ)
ask_agent("Giám đốc nhân sự có nghĩa vụ gì trong bộ quy tắc ứng xử kinh doanh?")

In [ ]:
# Test 3: Hybrid Search (Tình huống thực tế phức tạp cần kết hợp luật và sơ đồ)
ask_agent("Tôi ở bộ phận mua sắm. Anh trai tôi mở công ty văn phòng phẩm và muốn làm supplier cho công ty. Tôi có được quyết định chọn công ty của anh trai không và phải xin phép ai?")

## 5. Thống Kê GraphDB (Cypher)

In [ ]:
# Khởi tạo Graph
from langchain_community.graphs import Neo4jGraph
graph = Neo4jGraph()

# Xem tất cả các nhãn (Labels)
labels_query = """
MATCH (n)
UNWIND labels(n) as label
RETURN label, count(*) as count
ORDER BY count DESC
"""
print("=== THỐNG KÊ LABELS MỚI NHẤT ===")
for r in graph.query(labels_query):
    print(f"- {r[label]}: {r[count]} nodes")


## 6. Xem Số Lượng Relationships (Cypher)

In [ ]:
# Xem tất cả các loại Mối quan hệ (Relationships)
rels_query = """
MATCH ()-[r]->()
RETURN type(r) as relationship_type, count(*) as count
ORDER BY count DESC
"""
print("=== THỐNG KÊ RELATIONSHIPS ===")
for r in graph.query(rels_query):
    print(f"- {r[relationship_type]}: {r[count]} relationships")


## 7. Truy vấn Quan hệ 2 Node (Cypher)

In [ ]:
# Thử tìm các Node có quan hệ REQUIRES_APPROVAL_FROM
approval_query = """
MATCH (a)-[r:REQUIRES_APPROVAL_FROM]->(b)
RETURN labels(a)[0] AS Entity_A, a.id AS Name_A, 
       type(r) AS Relationship, 
       labels(b)[0] AS Entity_B, b.id AS Name_B
LIMIT 10
"""
print("=== DANH SÁCH DUYỆT ===")
for r in graph.query(approval_query):
    print(f"[{r["Name_A"]}] cần {r["Relationship"]} từ [{r["Name_B"]}]")
